# Tier 2 — Full-Data Experiments (Colab GPU)

Roda `CREDIT (30k)`, `ADULT (48k)` e `HIGGS50K` com os modelos escaláveis:
- NystromLSSVMColnorm
- FTTransformerCURColnorm
- FTTransformer (softmax / topk / entmax / sparsemax)

**Antes de rodar:** Faça `Runtime → Change runtime type → T4 GPU`

**Preparação:**
1. Compacte o projeto: `zip -r study.zip sparse-lssvm-transformers-study/`
2. Faça upload do `study.zip` para o Google Drive em `Meu Drive/dissertacao/`

In [ ]:
# ── Célula 1: Verificar GPU ───────────────────────────────────────────────────
!nvidia-smi
import torch
print(f"\nCUDA disponível: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memória: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# ── Célula 2: Montar Google Drive ────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

DRIVE_PATH = '/content/drive/MyDrive/dissertacao'
PROJECT_ZIP = f'{DRIVE_PATH}/study.zip'
PROJECT_DIR = '/content/study'

import os
print(f"Drive montado. Arquivos em dissertacao/:")
!ls '{DRIVE_PATH}'

In [ ]:
# ── Célula 3: Extrair projeto e baixar dados ──────────────────────────────────
import shutil, os

if os.path.exists(PROJECT_DIR):
    shutil.rmtree(PROJECT_DIR)

!unzip -q '{PROJECT_ZIP}' -d /content/
extracted = [d for d in os.listdir('/content/') if 'sparse-lssvm' in d or 'study' in d.lower()]
if extracted:
    os.rename(f'/content/{extracted[0]}', PROJECT_DIR)

os.chdir(PROJECT_DIR)
print(f"Diretório atual: {os.getcwd()}")

# O zip não inclui data/ (mantém tamanho <2 MB). Baixa agora.
print("\nBaixando datasets (CREDIT, ADULT, HIGGS50K)...")
!python scripts/download_data.py
print("\nDatasets prontos:")
!ls data/raw/

In [ ]:
# ── Célula 4: Instalar dependências ──────────────────────────────────────────
# Colab já tem: numpy, scipy, scikit-learn, torch
!pip install -q optuna entmax

# Verificar versões
import numpy, scipy, sklearn, torch, optuna
print(f"numpy {numpy.__version__} | scipy {scipy.__version__} | "
      f"sklearn {sklearn.__version__} | torch {torch.__version__} | "
      f"optuna {optuna.__version__}")

In [ ]:
# ── Célula 5: Restaurar resultados e params do Drive (retomar se necessário) ──
import shutil
from pathlib import Path

drive_results = Path(DRIVE_PATH)
local_results = Path('results')
local_tuning  = Path('results/tuning')
local_tuning.mkdir(parents=True, exist_ok=True)

# Restaura resultados de experimentos E params de tuning
for fname in ['tier2_full.json',
              'tuning/best_params_tier2.json']:
    src = drive_results / fname
    dst = local_results / fname
    if src.exists():
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy(src, dst)
        print(f'Restaurado: {fname}')
    else:
        print(f'Não encontrado (começando do zero): {fname}')

In [ ]:
# ── Célula 6: Rodar experimentos ──────────────────────────────────────────────
# --drive-path salva no Drive a cada 5 runs (proteção contra timeout)
# --folds 3 é mais rápido para datasets grandes (vs 5)
# --trials-ft 15 equilibra qualidade e velocidade no Colab

!python scripts/run_tier2_full.py \
    --seeds 30 \
    --trials-lssvm 50 \
    --trials-ft 15 \
    --folds 3 \
    --drive-path '{DRIVE_PATH}'

In [ ]:
# ── Célula 7: Verificar resultados ────────────────────────────────────────────
import json, numpy as np
from collections import defaultdict

results = json.load(open('results/tier2_full.json'))
print(f"Total de runs: {len(results)}")
print(f"Status ok: {sum(1 for r in results if r.get('status') == 'ok')}")
print(f"Erros: {sum(1 for r in results if r.get('status') != 'ok')}")
print()

scores = defaultdict(lambda: defaultdict(list))
for r in results:
    if r.get('status') != 'ok': continue
    model = r.get('model_variant') or r.get('model')
    scores[model][r['dataset']].append(r.get('f1_macro', float('nan')))

DATASETS = ['CREDIT', 'ADULT', 'HIGGS50K']
print(f"{'Model':<28}" + ''.join(f'{d:>10}' for d in DATASETS) + f"{'Mean':>8}")
print('-' * 66)
for model, ds_map in sorted(scores.items(),
                             key=lambda x: -np.nanmean([np.mean(v)
                             for v in x[1].values()])):
    vals = [np.mean(ds_map.get(d, [float('nan')])) for d in DATASETS]
    mean = np.nanmean(vals)
    print(f"{model:<28}" + ''.join(f'{v:10.4f}' if not np.isnan(v) else '         -'
                                   for v in vals) + f"{mean:8.4f}")

In [ ]:
# ── Célula 8: Salvar resultados finais no Drive ───────────────────────────────
import shutil
from pathlib import Path

drive_dest = Path(DRIVE_PATH)
drive_dest.mkdir(parents=True, exist_ok=True)

for fname in ['tier2_full.json', 'tuning/best_params_tier2.json']:
    src = Path('results') / fname
    dst = drive_dest / fname
    if src.exists():
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy(src, dst)
        print(f'Salvo: {dst}')

print('\nConteúdo do Drive:')
!ls -lh '{DRIVE_PATH}'